# 🦜🔗 LangChain: Interview Preparation & Code Reference Notebook

> **Purpose:** A complete reference guide — from fundamentals to production — for LangChain interview prep and quick code recall.

**Install dependencies:**
```bash
pip install langchain langchain-openai langchain-community faiss-cpu pydantic
```

---
**Sections:**
1. LangChain Overview
2. LLM Integration
3. Prompt Templates
4. LCEL (LangChain Expression Language)
5. Output Parsers
6. Chains
7. Document Loaders
8. Text Splitting
9. Embeddings
10. Vector Databases
11. Retrieval
12. RAG (Retrieval Augmented Generation)
13. Memory
14. Tools
15. Agents
16. Streaming Responses
17. Error Handling
18. Production Best Practices
19. Mini End-to-End RAG Project
20. Interview Questions & Answers

---
## ⚙️ Setup & Imports

In [ ]:
import os
from typing import List, Dict, Any, Optional

# ── Set your API key here (never hardcode in production) ──
os.environ["OPENAI_API_KEY"] = "your_key_here"

print("Environment configured.")

---
# 1. LangChain Overview

**What is LangChain?**  
LangChain is an open-source framework for building LLM-powered applications. It provides composable abstractions (chains, agents, retrievers, memory) that wire together LLMs, data sources, and tools.

**Core Components:**
| Component | Purpose |
|---|---|
| LLMs / Chat Models | Language model interface |
| Prompt Templates | Structured prompt management |
| Chains / LCEL | Composable pipelines |
| Retrievers / Vector Stores | External knowledge lookup |
| Agents + Tools | Dynamic reasoning + action |
| Memory | Conversation state |

**When to use LangChain:**
- RAG systems (question answering over documents)
- Conversational agents with tools
- Multi-step reasoning pipelines
- Structured data extraction from LLMs

**🎯 Interview Points:**
- LangChain v0.1+ introduced **LCEL** (LangChain Expression Language) — a declarative pipe-based syntax replacing legacy `LLMChain`.
- LangChain separates concerns: *orchestration* (chains/agents) from *model* (LLMs) from *data* (retrievers).
- `langchain-core` is the minimal dependency; `langchain-community` adds third-party integrations.

---
# 2. LLM Integration

**Definition:** LangChain wraps LLM providers (OpenAI, Anthropic, etc.) behind a unified interface. `ChatOpenAI` is the standard chat model — it accepts messages and returns a message response.

**Key parameters:** `model`, `temperature` (0=deterministic, 1=creative), `max_tokens`.

**🎯 Interview Points:**
- `temperature=0` → deterministic output; good for factual tasks.
- `ChatOpenAI` uses the **chat completions** API; `OpenAI` (legacy) uses completions API.
- All LangChain LLMs expose `.invoke()`, `.stream()`, and `.batch()` uniformly.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage


def initialize_llm(
    model: str = "gpt-4o-mini",
    temperature: float = 0.0,
    max_tokens: int = 512,
) -> ChatOpenAI:
    """Create and return a ChatOpenAI instance."""
    return ChatOpenAI(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
    )


def run_llm_prompt(llm: ChatOpenAI, user_prompt: str, system_prompt: str = "") -> str:
    """Send a prompt to the LLM and return the string response."""
    messages = []
    if system_prompt:
        messages.append(SystemMessage(content=system_prompt))
    messages.append(HumanMessage(content=user_prompt))

    response = llm.invoke(messages)  # returns AIMessage
    return response.content


# ── Example usage ──
llm = initialize_llm(temperature=0.2)
# answer = run_llm_prompt(llm, "What is the capital of France?", "You are a helpful assistant.")
# print(answer)
print("LLM initialized:", llm.model_name)

---
# 3. Prompt Templates

**Definition:** Prompt templates decouple prompt logic from input data. They accept variable placeholders and render a final prompt string or message list at runtime.

**Types:** `PromptTemplate` (single string), `ChatPromptTemplate` (message list for chat models).

**🎯 Interview Points:**
- Templates are *reusable* and *version-controllable* — key for production.
- `ChatPromptTemplate.from_messages()` takes `("role", "template")` tuples.
- `.format_messages()` renders the template; `.invoke()` returns a `PromptValue` usable in LCEL.

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate


def create_prompt_template(template_str: str, input_variables: List[str]) -> PromptTemplate:
    """Create a simple string PromptTemplate."""
    return PromptTemplate(
        template=template_str,
        input_variables=input_variables,
    )


def create_chat_prompt_template(system_msg: str, human_msg: str) -> ChatPromptTemplate:
    """Create a ChatPromptTemplate with system and human message slots."""
    return ChatPromptTemplate.from_messages([
        ("system", system_msg),
        ("human", human_msg),
    ])


def format_prompt(template: PromptTemplate, variables: Dict[str, str]) -> str:
    """Render a PromptTemplate with the given variable mapping."""
    return template.format(**variables)


# ── Example: Summarization prompt ──
summarization_prompt = create_chat_prompt_template(
    system_msg="You are a professional summarizer. Be concise and accurate.",
    human_msg="Summarize the following text in {num_sentences} sentence(s):\n\n{text}",
)

# Preview the rendered messages
rendered = summarization_prompt.format_messages(
    num_sentences=2,
    text="LangChain is a framework for developing LLM-powered applications.",
)
for msg in rendered:
    print(f"[{msg.type}]: {msg.content}")

---
# 4. LCEL — LangChain Expression Language

**Definition:** LCEL is a declarative syntax introduced in LangChain v0.1 that uses the `|` (pipe) operator to compose `Runnable` objects. Every LangChain component (prompt, LLM, parser) implements the `Runnable` interface.

**Core methods:** `.invoke(input)` · `.stream(input)` · `.batch(inputs)`

**🎯 Interview Points:**
- The `|` operator chains runnables: output of left feeds as input to right.
- LCEL enables **automatic parallelism** via `.batch()` and **streaming** with no extra code.
- `RunnablePassthrough` passes input unchanged — useful for injecting context in RAG.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def build_lcel_chain(
    prompt: ChatPromptTemplate,
    llm: ChatOpenAI,
    parser: StrOutputParser = None,
):
    """Compose a basic LCEL chain: prompt | llm | parser."""
    parser = parser or StrOutputParser()
    return prompt | llm | parser  # each | connects Runnables


def run_lcel_chain(chain, input_data: Dict[str, Any]) -> str:
    """Execute an LCEL chain with the provided input dictionary."""
    return chain.invoke(input_data)


# ── Example: joke chain ──
joke_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a witty comedian."),
    ("human", "Tell me a short joke about {topic}."),
])

joke_chain = build_lcel_chain(joke_prompt, llm)

# Uncomment to run:
# result = run_lcel_chain(joke_chain, {"topic": "Python programming"})
# print(result)
print("LCEL chain built:", joke_chain)

---
# 5. Output Parsers

**Definition:** Output parsers transform raw LLM text into structured Python objects. They sit at the end of LCEL chains and define the expected output schema.

**Types:** `StrOutputParser` (plain text), `JsonOutputParser` (dict), `PydanticOutputParser` (typed model).

**🎯 Interview Points:**
- `PydanticOutputParser` injects format instructions into the prompt automatically via `.get_format_instructions()`.
- Always include format instructions in the system/human prompt when using structured parsers.
- Output parsers implement `Runnable` so they plug directly into LCEL chains.

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain.output_parsers import PydanticOutputParser


# ── String parser ──
def parse_string_output() -> StrOutputParser:
    """Return a parser that extracts plain string content from AIMessage."""
    return StrOutputParser()


# ── Structured Pydantic model ──
class MovieReview(BaseModel):
    title: str = Field(description="Movie title")
    rating: float = Field(description="Rating from 0.0 to 10.0")
    summary: str = Field(description="One-sentence summary")


def build_structured_chain(llm: ChatOpenAI) -> Any:
    """Build an LCEL chain that returns a parsed MovieReview Pydantic object."""
    parser = PydanticOutputParser(pydantic_object=MovieReview)

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract movie review data. {format_instructions}"),
        ("human", "{review_text}"),
    ]).partial(format_instructions=parser.get_format_instructions())
    # .partial() pre-fills a template variable

    return prompt | llm | parser


def parse_structured_output(chain, review_text: str) -> MovieReview:
    """Run the structured chain and return a MovieReview instance."""
    return chain.invoke({"review_text": review_text})


# ── Example ──
structured_chain = build_structured_chain(llm)
# review = parse_structured_output(
#     structured_chain,
#     "Inception (2010) is a mind-bending masterpiece. I'd give it a 9.5/10."
# )
# print(review.title, review.rating)
print("Structured chain ready. Schema:", MovieReview.schema())

---
# 6. Chains

**Definition:** A chain sequences multiple LLM calls or processing steps. In modern LangChain, chains are built with LCEL. The classic pattern is: `Prompt → LLM → Parser`.

**Sequential chains** pass output of one step as input to the next, enabling multi-hop reasoning.

**🎯 Interview Points:**
- Legacy `LLMChain` and `SequentialChain` are deprecated — prefer LCEL.
- Use `RunnablePassthrough.assign()` to merge outputs into a running context dict.
- Chains are composable: `chain_a | chain_b` creates a new chain.

In [ ]:
from langchain_core.runnables import RunnablePassthrough


def create_simple_chain(llm: ChatOpenAI) -> Any:
    """Simple chain: question → answer."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer the question clearly and concisely."),
        ("human", "{question}"),
    ])
    return prompt | llm | StrOutputParser()


def create_sequential_chain(llm: ChatOpenAI) -> Any:
    """
    Two-step sequential chain:
      Step 1: Generate reasoning for the question.
      Step 2: Produce a final concise answer from that reasoning.
    """
    reasoning_prompt = ChatPromptTemplate.from_messages([
        ("system", "Think step-by-step about the question."),
        ("human", "Question: {question}"),
    ])

    answer_prompt = ChatPromptTemplate.from_messages([
        ("system", "Given this reasoning, provide a short final answer."),
        ("human", "Reasoning: {reasoning}\nQuestion: {question}"),
    ])

    # Step 1: produce reasoning, keep original question in context
    step1 = RunnablePassthrough.assign(
        reasoning=reasoning_prompt | llm | StrOutputParser()
    )
    # Step 2: use both question + reasoning to produce final answer
    step2 = answer_prompt | llm | StrOutputParser()

    return step1 | step2


def run_chain(chain, input_data: Dict[str, Any]) -> str:
    """Generic chain runner."""
    return chain.invoke(input_data)


# ── Example ──
simple_chain = create_simple_chain(llm)
seq_chain = create_sequential_chain(llm)

# answer = run_chain(seq_chain, {"question": "Why is the sky blue?"})
# print(answer)
print("Simple chain:", simple_chain)
print("Sequential chain ready.")

---
# 7. Document Loaders

**Definition:** Document loaders ingest data from various sources (files, URLs, databases) and return a list of `Document` objects — each with `.page_content` (string) and `.metadata` (dict).

**Common loaders:** `TextLoader`, `PyPDFLoader`, `WebBaseLoader`, `CSVLoader`.

**🎯 Interview Points:**
- `Document` is the fundamental data unit in LangChain's retrieval pipeline.
- Loaders abstract away parsing — swap sources without changing downstream code.
- `.load()` returns `List[Document]`; `.lazy_load()` returns a generator (memory-efficient).

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
import tempfile, os


def load_documents(file_path: str) -> List[Document]:
    """Load documents from a text file using TextLoader."""
    loader = TextLoader(file_path, encoding="utf-8")
    return loader.load()  # returns List[Document]


def load_documents_from_string(text: str) -> List[Document]:
    """Helper: wrap a raw string as a Document for testing."""
    return [Document(page_content=text, metadata={"source": "inline"})]


def inspect_documents(docs: List[Document]) -> None:
    """Print summary info about loaded documents."""
    print(f"Total documents: {len(docs)}")
    for i, doc in enumerate(docs):
        print(f"  [{i}] chars={len(doc.page_content)} | metadata={doc.metadata}")


# ── Example: write a temp file, load it ──
sample_text = """LangChain makes it easy to build LLM applications.
It supports RAG, agents, and multi-step reasoning pipelines.
The framework is modular and integrates with many LLM providers."""

# Write temp file
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8") as f:
    f.write(sample_text)
    tmp_path = f.name

docs = load_documents(tmp_path)
inspect_documents(docs)
os.unlink(tmp_path)  # cleanup

# PDF loader (requires pypdf):
# from langchain_community.document_loaders import PyPDFLoader
# def load_pdf(pdf_path: str) -> List[Document]:
#     return PyPDFLoader(pdf_path).load()

---
# 8. Text Splitting

**Definition:** Text splitters divide long documents into smaller chunks before embedding. This is critical for RAG because LLMs have context limits and embedding models work best on short, focused passages.

**`RecursiveCharacterTextSplitter`** tries to split on `\n\n`, `\n`, ` `, `` — in priority order — to keep semantic units intact.

**🎯 Interview Points:**
- `chunk_overlap` prevents loss of context at chunk boundaries (typically 10–20% of chunk size).
- Smaller chunks → more precise retrieval but more vectors. Larger chunks → more context but noisier.
- Always split *after* loading, *before* embedding.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(
    docs: List[Document],
    chunk_size: int = 500,
    chunk_overlap: int = 50,
) -> List[Document]:
    """
    Split documents into chunks using RecursiveCharacterTextSplitter.
    chunk_size: max characters per chunk.
    chunk_overlap: shared characters between adjacent chunks.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],  # tries these in order
    )
    return splitter.split_documents(docs)


# ── Example ──
long_text = " ".join(["LangChain is great for RAG applications."] * 30)
raw_docs = [Document(page_content=long_text, metadata={"source": "example"})]

chunks = split_documents(raw_docs, chunk_size=200, chunk_overlap=20)
print(f"Original doc length: {len(long_text)} chars")
print(f"Number of chunks: {len(chunks)}")
print(f"First chunk preview: '{chunks[0].page_content[:80]}...'") if chunks else None

---
# 9. Embeddings

**Definition:** Embeddings convert text into dense numeric vectors (e.g., 1536-dim for `text-embedding-ada-002`). Semantically similar texts produce vectors close together in the embedding space.

**Use case:** Store document chunk embeddings in a vector DB; at query time, embed the question and find nearest chunks.

**🎯 Interview Points:**
- Embeddings are *separate* from the generative LLM — you can mix and match providers.
- `.embed_documents(texts)` → `List[List[float]]`; `.embed_query(text)` → `List[float]`.
- `text-embedding-3-small` is OpenAI's latest efficient model (cheaper than ada-002).

In [ ]:
from langchain_openai import OpenAIEmbeddings


def create_embeddings(model: str = "text-embedding-3-small") -> OpenAIEmbeddings:
    """Initialize an OpenAI embedding model."""
    return OpenAIEmbeddings(model=model)


def embed_query_text(embeddings: OpenAIEmbeddings, query: str) -> List[float]:
    """Embed a single query string; returns a float vector."""
    return embeddings.embed_query(query)


def embed_documents(embeddings: OpenAIEmbeddings, texts: List[str]) -> List[List[float]]:
    """Embed a batch of document strings."""
    return embeddings.embed_documents(texts)


# ── Example ──
embedding_model = create_embeddings()

# Uncomment to run (requires API key):
# vec = embed_query_text(embedding_model, "What is RAG?")
# print(f"Vector dimensions: {len(vec)}")
# print(f"First 5 values: {vec[:5]}")

print("Embedding model initialized:", embedding_model.model)

---
# 10. Vector Databases

**Definition:** Vector databases store document embeddings and support fast **approximate nearest-neighbor (ANN)** search. FAISS (Facebook AI Similarity Search) is a popular in-memory option for development.

**Production alternatives:** Pinecone, Weaviate, Chroma, Qdrant, pgvector.

**🎯 Interview Points:**
- `FAISS.from_documents()` embeds and indexes in one call.
- FAISS is in-memory — not persistent by default. Use `.save_local()` / `.load_local()` for persistence.
- Similarity search uses **cosine similarity** or **L2 distance** depending on index type.

In [ ]:
from langchain_community.vectorstores import FAISS


def create_vector_store(
    docs: List[Document],
    embeddings: OpenAIEmbeddings,
) -> FAISS:
    """Embed documents and store in a FAISS index."""
    return FAISS.from_documents(docs, embeddings)  # embeds + indexes


def store_documents(
    vector_store: FAISS,
    new_docs: List[Document],
    embeddings: OpenAIEmbeddings,
) -> FAISS:
    """Add more documents to an existing FAISS vector store."""
    new_store = FAISS.from_documents(new_docs, embeddings)
    vector_store.merge_from(new_store)  # merge indexes
    return vector_store


def search_documents(
    vector_store: FAISS,
    query: str,
    k: int = 3,
) -> List[Document]:
    """Run similarity search and return top-k relevant documents."""
    return vector_store.similarity_search(query, k=k)


# ── Example (requires API key) ──
sample_docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="FAISS is a fast similarity search library."),
    Document(page_content="RAG combines retrieval with generation."),
]

# vector_store = create_vector_store(sample_docs, embedding_model)
# results = search_documents(vector_store, "What is retrieval augmented generation?")
# for r in results:
#     print(r.page_content)

print("FAISS vector store functions defined.")

---
# 11. Retrieval

**Definition:** A **retriever** is the abstraction over a vector store that fetches relevant documents given a query. It exposes a clean `.invoke(query)` interface (returns `List[Document]`), hiding the underlying storage details.

**Common retrieval strategies:** similarity search, MMR (maximal marginal relevance), BM25 hybrid search.

**🎯 Interview Points:**
- Retrievers implement `Runnable` → plug directly into LCEL chains.
- `search_type="mmr"` reduces redundancy in results by penalizing near-duplicate chunks.
- `search_kwargs={"k": N}` controls how many documents are returned (top-k).

In [ ]:
from langchain_core.vectorstores import VectorStoreRetriever


def create_retriever(
    vector_store: FAISS,
    k: int = 4,
    search_type: str = "similarity",
) -> VectorStoreRetriever:
    """
    Convert a vector store into a retriever.
    search_type: 'similarity' | 'mmr'
    """
    return vector_store.as_retriever(
        search_type=search_type,
        search_kwargs={"k": k},
    )


def retrieve_context(retriever: VectorStoreRetriever, query: str) -> List[Document]:
    """Retrieve relevant documents for a query."""
    return retriever.invoke(query)


def format_docs_as_context(docs: List[Document]) -> str:
    """Concatenate document content into a single context string for prompt injection."""
    return "\n\n".join(doc.page_content for doc in docs)


# ── Example ──
# retriever = create_retriever(vector_store, k=3)
# docs = retrieve_context(retriever, "What is RAG?")
# context = format_docs_as_context(docs)
# print(context)

print("Retriever functions defined.")

---
# 12. RAG — Retrieval Augmented Generation

**Definition:** RAG is a pattern where a retriever fetches relevant document chunks from a knowledge base, and those chunks are injected into the prompt as context before the LLM generates an answer. This grounds the LLM in external knowledge without retraining.

**Pipeline:** `Query → Retriever → Context Docs → Prompt + Context → LLM → Answer`

**🎯 Interview Points:**
- RAG solves the **hallucination problem** by grounding answers in retrieved evidence.
- The key insight: separate *storage* of knowledge from the generative model.
- `RunnablePassthrough` is used to pass the original question alongside the retrieved context.

In [ ]:
from langchain_core.runnables import RunnablePassthrough


def build_rag_pipeline(
    retriever: VectorStoreRetriever,
    llm: ChatOpenAI,
) -> Any:
    """
    Build an LCEL RAG pipeline:
      {context, question} → prompt → llm → string answer
    """
    rag_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are an expert assistant. Answer using ONLY the provided context."
         " If unsure, say 'I don't know'.\n\nContext:\n{context}"),
        ("human", "{question}"),
    ])

    # RunnablePassthrough.assign injects retrieved context into input dict
    rag_chain = (
        RunnablePassthrough.assign(
            context=lambda x: format_docs_as_context(retriever.invoke(x["question"]))
        )
        | rag_prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain


def run_rag_query(rag_chain, question: str) -> str:
    """Run a question through the RAG pipeline."""
    return rag_chain.invoke({"question": question})


# ── Example ──
# rag_chain = build_rag_pipeline(retriever, llm)
# answer = run_rag_query(rag_chain, "What is RAG?")
# print(answer)

print("RAG pipeline functions defined.")

---
# 13. Memory

**Definition:** Memory components persist conversation history across turns, allowing the LLM to reference earlier messages. `ConversationBufferMemory` stores the full history as a list of messages.

**Note:** In LCEL, memory is managed manually by maintaining a `chat_history` list and passing it to the chain input.

**🎯 Interview Points:**
- `ConversationBufferWindowMemory(k=N)` keeps only last N turns — avoids context overflow.
- `ConversationSummaryMemory` summarizes old history to save tokens.
- For production, persist history to Redis/Postgres rather than in-memory.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_core.prompts import MessagesPlaceholder


def initialize_memory() -> List[BaseMessage]:
    """Return an empty chat history list (LCEL-style memory)."""
    return []


def build_memory_chain(llm: ChatOpenAI) -> Any:
    """Build a chain that accepts chat_history + user input."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Remember the conversation context."),
        MessagesPlaceholder(variable_name="chat_history"),  # inject history here
        ("human", "{input}"),
    ])
    return prompt | llm | StrOutputParser()


def run_chat_with_memory(
    chain,
    history: List[BaseMessage],
    user_input: str,
) -> tuple[str, List[BaseMessage]]:
    """
    Run one turn of conversation.
    Returns (assistant_reply, updated_history).
    """
    response = chain.invoke({"input": user_input, "chat_history": history})

    # Append turn to history
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))
    return response, history


# ── Example ──
memory_chain = build_memory_chain(llm)
history = initialize_memory()

# reply1, history = run_chat_with_memory(memory_chain, history, "My name is Alex.")
# reply2, history = run_chat_with_memory(memory_chain, history, "What's my name?")
# print(reply2)  # Should recall "Alex"

print(f"Memory chain ready. History length: {len(history)}")

---
# 14. Tools

**Definition:** Tools are callable Python functions exposed to LLM agents. The agent decides which tool to call based on the task. LangChain provides the `@tool` decorator to wrap any function as a tool with auto-generated schema.

**Key properties:** `name`, `description` (LLM reads this to decide when to use it), `args_schema`.

**🎯 Interview Points:**
- The `@tool` decorator auto-extracts the docstring as the tool description — write clear docstrings!
- Tools must have typed arguments so the LLM can generate correct JSON to call them.
- `StructuredTool.from_function()` supports multi-argument tools with Pydantic schemas.

In [ ]:
from langchain_core.tools import tool


@tool
def calculator_tool(expression: str) -> str:
    """
    Evaluate a mathematical expression string and return the result.
    Input: a valid Python math expression like '2 + 2' or '10 * 3.5'.
    """
    try:
        # eval is acceptable here in controlled tool context
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


@tool
def weather_tool(city: str) -> str:
    """
    Get the current weather for a given city.
    Returns a mock weather report (replace with real API in production).
    """
    mock_data = {
        "london": "Cloudy, 14°C",
        "new york": "Sunny, 22°C",
        "tokyo": "Rainy, 18°C",
    }
    return mock_data.get(city.lower(), f"No weather data available for {city}.")


def create_tool_list() -> list:
    """Return the list of available tools for an agent."""
    return [calculator_tool, weather_tool]


def execute_tool(tool_fn, **kwargs) -> str:
    """Directly invoke a tool by name with keyword arguments."""
    return tool_fn.invoke(kwargs)


# ── Example ──
print(calculator_tool.name, "|", calculator_tool.description[:60])
print(execute_tool(calculator_tool, expression="15 * 4 + 7"))
print(execute_tool(weather_tool, city="London"))

---
# 15. Agents

**Definition:** An agent uses an LLM as a reasoning engine to dynamically decide *which tool to call* and *what arguments to use*, based on the user's goal. The **ReAct** pattern (Reason + Act) is the most common agent strategy.

**Flow:** `Thought → Action (tool call) → Observation → Thought → ... → Final Answer`

**🎯 Interview Points:**
- `create_tool_calling_agent` uses the model's native function-calling capability (more reliable than text parsing).
- `AgentExecutor` is the runtime loop — handles tool calls, retries, and stopping conditions.
- `max_iterations` prevents infinite loops; `handle_parsing_errors=True` adds robustness.

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor


def create_agent(
    llm: ChatOpenAI,
    tools: list,
    system_message: str = "You are a helpful assistant with access to tools.",
) -> AgentExecutor:
    """
    Create a tool-calling agent with an AgentExecutor wrapper.
    Uses the model's native function-calling API.
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_message),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),  # required by agent
    ])

    agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

    return AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=True,           # prints Thought/Action/Observation logs
        max_iterations=5,       # safety limit
        handle_parsing_errors=True,
    )


def run_agent_query(agent_executor: AgentExecutor, query: str) -> str:
    """Run a user query through the agent and return the final answer."""
    result = agent_executor.invoke({"input": query})
    return result["output"]


# ── Example ──
tools = create_tool_list()
agent_executor = create_agent(llm, tools)

# answer = run_agent_query(agent_executor, "What is 137 * 42? Also, what's the weather in Tokyo?")
# print(answer)

print(f"Agent created with {len(tools)} tools.")

---
# 16. Streaming Responses

**Definition:** Streaming delivers tokens to the client as they are generated, rather than waiting for the complete response. This drastically improves perceived latency in user-facing applications.

**`chain.stream(input)`** returns a generator of string chunks (with `StrOutputParser`).

**🎯 Interview Points:**
- All LCEL chains support `.stream()` automatically — no special setup needed.
- In production, stream over WebSocket or Server-Sent Events (SSE) to the frontend.
- `.astream()` is the async variant for use in FastAPI or async frameworks.

In [ ]:
from typing import Generator


def stream_llm_response(
    chain,
    input_data: Dict[str, Any],
) -> Generator[str, None, None]:
    """Stream tokens from an LCEL chain, yielding each chunk as it arrives."""
    for chunk in chain.stream(input_data):
        yield chunk


def stream_and_print(chain, input_data: Dict[str, Any]) -> str:
    """Stream response and print each token; returns full assembled response."""
    full_response = ""
    print("Streaming response: ", end="", flush=True)
    for chunk in stream_llm_response(chain, input_data):
        print(chunk, end="", flush=True)  # print without newline
        full_response += chunk
    print()  # final newline
    return full_response


# ── Async variant (for FastAPI/async contexts) ──
async def astream_llm_response(chain, input_data: Dict[str, Any]):
    """Async streaming — use with 'async for chunk in astream_llm_response(...)'"""
    async for chunk in chain.astream(input_data):
        yield chunk


# ── Example ──
# stream_and_print(joke_chain, {"topic": "databases"})

print("Streaming functions defined. Call stream_and_print(chain, input) to use.")

---
# 17. Error Handling

**Definition:** LLM APIs can fail due to rate limits, network errors, or invalid responses. Production pipelines need retry logic, validation, and graceful fallbacks.

**`with_retry()`** is LangChain's built-in retry wrapper for any Runnable.

**🎯 Interview Points:**
- `llm.with_retry(stop_after_attempt=3)` wraps the LLM with automatic retries.
- `llm.with_fallbacks([backup_llm])` falls back to a different model on failure.
- Always validate Pydantic output — LLM might return malformed JSON despite instructions.

In [ ]:
from langchain_core.runnables import RunnableWithFallbacks
import time


def build_resilient_llm(
    primary_llm: ChatOpenAI,
    max_retries: int = 3,
) -> Any:
    """Wrap an LLM with automatic retry on transient errors."""
    return primary_llm.with_retry(
        stop_after_attempt=max_retries,
        wait_exponential_jitter=True,  # exponential backoff with jitter
    )


def build_llm_with_fallback(
    primary_llm: ChatOpenAI,
    fallback_model: str = "gpt-3.5-turbo",
) -> RunnableWithFallbacks:
    """Fall back to a cheaper/different model if the primary fails."""
    fallback_llm = initialize_llm(model=fallback_model)
    return primary_llm.with_fallbacks([fallback_llm])


def safe_llm_call(
    chain,
    input_data: Dict[str, Any],
    default_response: str = "Unable to process request.",
) -> str:
    """Execute a chain with a try/except guard; return default on failure."""
    try:
        return chain.invoke(input_data)
    except Exception as e:
        print(f"[ERROR] Chain invocation failed: {type(e).__name__}: {e}")
        return default_response


# ── Example ──
resilient_llm = build_resilient_llm(llm, max_retries=3)
print("Resilient LLM wrapper created.")

# Test safe_llm_call with a broken chain
result = safe_llm_call(joke_chain, {"wrong_key": "value"}, default_response="fallback")
print(f"Safe call result: '{result}'")

---
# 18. Production Best Practices

Key principles for deploying LangChain applications reliably at scale:

## 🔧 Observability
- Use **LangSmith** (Anthropic's tracing platform) to trace every chain/agent call: set `LANGCHAIN_TRACING_V2=true` and `LANGCHAIN_API_KEY`.
- Log inputs, outputs, latency, and token counts for every LLM call.

## 📝 Prompt Versioning
- Store prompts in a hub (LangSmith Prompt Hub) or a config file with version IDs.
- Never hardcode prompts inline in application code.

## ⚡ Caching
- Use `langchain.globals.set_llm_cache(InMemoryCache())` or Redis cache to avoid re-calling the LLM for identical inputs.
- Semantic caching (embedding-based) catches near-duplicate queries.

## 🛡️ Guardrails
- Validate LLM outputs with Pydantic before using them.
- Use input/output moderation layers (OpenAI Moderation API, Llama Guard).
- Add `RunnableLambda` validators in the chain to check output quality.

## 🚦 Rate Limiting
- Use `asyncio.Semaphore` or a token bucket algorithm to throttle concurrent LLM calls.
- Set sensible `max_tokens` limits to control cost.
- Use `chain.batch(inputs, config={"max_concurrency": 5})` for controlled parallelism.

In [ ]:
import langchain
from langchain.cache import InMemoryCache
from langchain_core.runnables import RunnableLambda


def enable_llm_cache() -> None:
    """Enable in-memory caching for all LLM calls globally."""
    langchain.llm_cache = InMemoryCache()
    print("LLM in-memory cache enabled.")


def configure_langsmith_tracing(
    api_key: str = "your_langsmith_key_here",
    project_name: str = "my-langchain-app",
) -> None:
    """Configure LangSmith tracing environment variables."""
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_API_KEY"] = api_key
    os.environ["LANGCHAIN_PROJECT"] = project_name
    print(f"LangSmith tracing configured for project: {project_name}")


def add_output_validator(chain, validator_fn) -> Any:
    """
    Append a validation step to a chain using RunnableLambda.
    validator_fn should raise ValueError on invalid output.
    """
    return chain | RunnableLambda(validator_fn)


# ── Example validator ──
def check_non_empty(text: str) -> str:
    if not text or len(text.strip()) < 5:
        raise ValueError(f"Output too short: '{text}'")
    return text


validated_chain = add_output_validator(joke_chain, check_non_empty)
enable_llm_cache()
print("Production wrappers configured.")

---
# 19. Mini End-to-End Project: RAG Question Answering System

**Project:** A complete, runnable RAG QA system that:
1. Takes a list of text documents
2. Splits them into chunks
3. Embeds and stores in FAISS
4. Creates a retriever
5. Builds an LCEL RAG chain
6. Answers questions grounded in the documents

**Pipeline:** `Documents → Splitter → Embeddings → FAISS → Retriever → Prompt → LLM → Answer`

In [ ]:
from dataclasses import dataclass


@dataclass
class RAGSystem:
    """Encapsulates all components of a RAG QA system."""
    vector_store: FAISS
    retriever: VectorStoreRetriever
    chain: Any


def build_rag_system(
    documents: List[str],
    llm: ChatOpenAI,
    embeddings: OpenAIEmbeddings,
    chunk_size: int = 400,
    chunk_overlap: int = 40,
    top_k: int = 3,
) -> RAGSystem:
    """
    Build a complete RAG system from raw text strings.

    Steps:
      1. Wrap strings as Documents
      2. Split into chunks
      3. Embed and index in FAISS
      4. Create retriever
      5. Build LCEL RAG chain
    """
    # Step 1: Wrap raw text
    raw_docs = [Document(page_content=t, metadata={"idx": i}) for i, t in enumerate(documents)]

    # Step 2: Split
    chunks = split_documents(raw_docs, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    print(f"[RAG] Split {len(raw_docs)} docs → {len(chunks)} chunks")

    # Step 3: Embed + index
    vector_store = create_vector_store(chunks, embeddings)
    print(f"[RAG] Indexed {len(chunks)} chunks in FAISS")

    # Step 4: Retriever
    retriever = create_retriever(vector_store, k=top_k)

    # Step 5: RAG chain
    chain = build_rag_pipeline(retriever, llm)

    return RAGSystem(vector_store=vector_store, retriever=retriever, chain=chain)


def ask_question(rag_system: RAGSystem, question: str) -> str:
    """Ask a question and get a grounded answer from the RAG system."""
    return run_rag_query(rag_system.chain, question)


# ── Demo documents ──
DEMO_DOCUMENTS = [
    """LangChain is a framework for developing applications powered by large language models.
    It provides tools for chaining LLM calls, managing prompts, integrating with vector databases,
    and building agents that can use external tools.""",

    """FAISS (Facebook AI Similarity Search) is an efficient library for similarity search
    and clustering of dense vectors. It is widely used in production RAG systems as an
    in-memory vector store that supports cosine similarity and L2 distance searches.""",

    """Retrieval Augmented Generation (RAG) is an AI architecture that combines a retrieval
    system with a generative language model. Instead of relying solely on parametric knowledge,
    RAG fetches relevant document chunks from a knowledge base and injects them into the
    LLM prompt, dramatically reducing hallucinations and enabling domain-specific QA.""",

    """LangChain Expression Language (LCEL) is the modern way to compose LangChain pipelines.
    Using the pipe operator (|), developers chain Runnables together declaratively.
    LCEL automatically supports streaming, batching, async execution, and tracing.""",
]

# ── Run the system (requires API key) ──
print("RAG system builder defined. To run:")
print()
print("  rag = build_rag_system(DEMO_DOCUMENTS, llm, embedding_model)")
print("  answer = ask_question(rag, 'What is RAG and why does it reduce hallucinations?')")
print("  print(answer)")

# ── Uncomment below to actually run ──
# rag = build_rag_system(DEMO_DOCUMENTS, llm, embedding_model)
# questions = [
#     "What is LangChain used for?",
#     "How does FAISS work?",
#     "Why does RAG reduce hallucinations?",
#     "What is LCEL?",
# ]
# for q in questions:
#     print(f"Q: {q}")
#     print(f"A: {ask_question(rag, q)}")
#     print()

---
# 20. Interview Questions & Answers

> Quick reference for the most commonly asked LangChain interview questions.

---

### Q1: What is LCEL (LangChain Expression Language)?
**A:** LCEL is a declarative syntax introduced in LangChain v0.1 that composes pipeline steps using the `|` (pipe) operator. Every component (prompt, LLM, parser, retriever) implements the `Runnable` interface, enabling uniform `.invoke()`, `.stream()`, and `.batch()` calls. LCEL automatically supports streaming, async, and parallelism without extra code.

---

### Q2: What is the difference between a Chain and an Agent?
**A:** A **Chain** follows a *fixed, predefined sequence* of steps — the flow is determined at build time (e.g., prompt → LLM → parser). An **Agent** uses the LLM as a *dynamic reasoning engine* that decides at runtime which tools to call and in what order, based on the user's goal. Chains are faster and more predictable; agents are more flexible but less deterministic.

---

### Q3: Why do we chunk documents before embedding?
**A:** Three reasons: (1) embedding models have token limits (e.g., 8192 for OpenAI), (2) smaller, focused chunks produce more semantically precise embeddings, improving retrieval accuracy, and (3) injecting entire documents into a prompt exceeds the LLM's context window. `chunk_overlap` is used to avoid losing context at chunk boundaries.

---

### Q4: What is RAG and why is it useful?
**A:** RAG (Retrieval Augmented Generation) augments an LLM's response with context retrieved from an external knowledge base. At query time, relevant document chunks are fetched via semantic search and injected into the prompt. This reduces hallucinations, enables up-to-date knowledge without retraining, and allows domain-specific question answering over private data.

---

### Q5: What is a Retriever in LangChain?
**A:** A Retriever is an abstraction over a vector store that exposes a single `.invoke(query) → List[Document]` interface. It hides the underlying storage engine (FAISS, Pinecone, etc.). Created via `vector_store.as_retriever()`, it supports different search strategies: `similarity`, `mmr` (max marginal relevance), and threshold-based filtering.

---

### Q6: What is `RunnablePassthrough` and when do you use it?
**A:** `RunnablePassthrough` passes its input unchanged downstream. In RAG chains, it's used with `.assign()` to inject retrieved context into the input dict without losing the original question: `RunnablePassthrough.assign(context=retriever)`. This allows the downstream prompt to access both `{question}` and `{context}`.

---

### Q7: How does memory work in modern LangChain (LCEL)?
**A:** In LCEL, memory is managed explicitly: you maintain a `chat_history` list of `HumanMessage`/`AIMessage` objects and pass it into the chain on each turn via a `MessagesPlaceholder`. After each turn, you append the new messages to the list. This is more transparent than legacy `ConversationBufferMemory` and works naturally with LCEL's stateless design.

---

### Q8: What is the difference between `StrOutputParser` and `PydanticOutputParser`?
**A:** `StrOutputParser` simply extracts the `.content` string from an `AIMessage`. `PydanticOutputParser` parses the LLM's text output into a typed Pydantic model — it automatically generates format instructions (injected into the prompt) that tell the LLM exactly what JSON schema to produce, then validates and parses the response.

---

### Q9: How do you handle LLM errors and rate limits in production?
**A:** Use `llm.with_retry(stop_after_attempt=3, wait_exponential_jitter=True)` for automatic retries with exponential backoff. Use `llm.with_fallbacks([backup_llm])` to switch models on failure. Wrap chain calls in try/except for graceful degradation. For rate limits specifically, use `chain.batch(inputs, config={"max_concurrency": 5})` to control parallel requests.

---

### Q10: What is the `@tool` decorator and what makes a good tool?
**A:** `@tool` wraps a Python function as a LangChain Tool, auto-extracting the function name and docstring as the tool's `name` and `description`. The agent uses the description to decide when to call the tool. A good tool has: (1) a clear, specific docstring explaining *exactly* what it does and what input format it expects, (2) typed parameters so the LLM can generate correct JSON, and (3) robust error handling since the agent depends on tool output to continue reasoning.

---

## 📌 Quick Cheat Sheet

| Concept | Key Class/Function | One-liner |
|---|---|---|
| Chat LLM | `ChatOpenAI` | LLM wrapper with `.invoke()` |
| Prompt | `ChatPromptTemplate` | Parameterized message templates |
| LCEL | `prompt \| llm \| parser` | Declarative pipe composition |
| Structured output | `PydanticOutputParser` | LLM → typed Python object |
| Document | `Document` | `page_content` + `metadata` |
| Splitter | `RecursiveCharacterTextSplitter` | Chunk docs for embedding |
| Embeddings | `OpenAIEmbeddings` | Text → float vector |
| Vector Store | `FAISS` | Index + ANN search |
| Retriever | `.as_retriever()` | Query → `List[Document]` |
| RAG | `RunnablePassthrough.assign` | Inject context into prompt |
| Memory | `MessagesPlaceholder` | Manual chat history injection |
| Tool | `@tool` decorator | Function the agent can call |
| Agent | `AgentExecutor` | LLM + tools runtime loop |
| Streaming | `.stream()` / `.astream()` | Token-by-token delivery |
| Retry | `.with_retry()` | Auto-retry on failure |
| Tracing | LangSmith + `LANGCHAIN_TRACING_V2` | Full observability |